# Text Representation

Name : Maan A. Alghamdi  
ID : 2240001433

### Setup
I keep `simpsons_script_lines.csv` in the same working folder as this notebook and run the cells from top to bottom. If dependencies are missing, I run `%pip install pandas numpy scikit-learn gensim nltk` in a notebook cell, then restart the kernel. No NLTK data download is required.

Text representation in NLP means converting text into a numerical form that a machine learning model can understand and process.

## What is Embedding?

Embeddings in NLP is a technique where individual words are represented as real-valued vectors and captures inter-word semantics.


In this notebook, We going to intreduce 2 techniques for embedding. These techniques will be used for a machine learning models such as SVM, Random forest, ... ect.

<img src="https://drive.google.com/uc?export=view&id=1jd0u_sGppDKqBYbhtJfGxp14lnUWUTTw" width="900">

# 1- TF-IDF

<img src="https://drive.google.com/uc?export=view&id=1saSt0mMgOQ2ybbTms1lu_ruhUcBWkKzL" width="500">

TF-IDF stands for term frequency-inverse document frequency. It is a measure that discounts common words. Used in the fields of information retrieval (IR) and machine learning, that can quantify the importance of words in a document amongst a collection of documents (also known as a corpus).

### Components of TF-IDF

1. **TF (Term Frequency):**  
   Measures how frequently a term appears in a document.


$$ \text{TF}(t, d) = \frac{\text{Number of times term } t \text{ appears in document } d}{\text{Total number of terms in document } d} $$


2. **IDF (Inverse Document Frequency):**  
   Measures how important a term is across all documents. Words that appear in many documents get lower scores.

$$    \text{IDF}(t) = \log \left(\frac{\text{Number of all documents N}}{\text{Number of documents containing the term } t}\right) $$

<img src="https://drive.google.com/uc?export=view&id=1JqPILC8TTh3yDQZCSuPNXwm6ER5YeJFh" width="900">

In [1]:
import pandas as pd
import numpy as np
import re
import zlib
from pathlib import Path
import gensim
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

# A stable hash and a single worker make training repeatable across runs.
def stable_hash(word):
    return zlib.crc32(word.encode("utf-8"))

In [2]:
doc_1 = "Data is the oil of the digital economy"
doc_2 = "Data is a new oil"

data = [doc_1, doc_2]


In [3]:
tfidf = TfidfVectorizer()
result = tfidf.fit_transform(data) # returns sparce matrix

In [4]:
df = pd.DataFrame(result.toarray(), columns=tfidf.get_feature_names_out())
df

,data,digital,economy,is,new,of,oil,the
0,0.243777,0.34262,0.34262,0.243777,0.000000,0.34262,0.243777,0.68524
1,0.448321,0.00000,0.00000,0.448321,0.630099,0.00000,0.448321,0.00000


# Cosine similarity

In NLP, Cosine similarity is a metric used to measure how similar the documents are.


$$ \text{cosine similarity} = \frac{A \cdot B}{\|A\| \times \|B\|} $$

Where:  
$ A \cdot B $  = dot product of vectors A and B  
$ \|A\| $ = magnitude (length) of vector A  
$ \|B\| $ =  magnitude (length) of vector B

#### Intuition

- If vectors point in the **same direction**, cosine similarity = **1** (maximum similarity).
- If vectors are **orthogonal (90° apart)**, cosine similarity = **0** (no similarity).

<img src="https://drive.google.com/uc?export=view&id=1b-o8CVfHBsjUGY_hXmxevU91gHidIuxO" width="900">

In [5]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
# Sample text data (replace with your own documents)

doc_1 = "Data is the oil of the digital economy"
doc_2 = "Data is a new oil"

data = [doc_1, doc_2]

In [7]:
tfidf_vectorizer = TfidfVectorizer()
vector_matrix = tfidf_vectorizer.fit_transform(data)

In [8]:
# Calculate the cosine similarity between the documents
cosine_similarity_matrix = cosine_similarity(vector_matrix)

df_cosine = pd.DataFrame(data=cosine_similarity_matrix, index=data, columns=data)

df_cosine

,Data is the oil of the digital economy,Data is a new oil
Data is the oil of the digital economy,1.000000,0.327871
Data is a new oil,0.327871,1.000000


# 2- What is Word2vec?

Word2Vec consists of models for generating word embedding. These models are two-layer neural networks having one input layer, one hidden layer, and one output layer.

Word2Vec utilizes two architectures :
1. CBOW (Continuous Bag of Words)
2. **Skip Gram**
<img src="https://drive.google.com/uc?export=view&id=1K38nEu_KhgtSJuG2RySwc6U5AAjVAgWZ" width="600">


Run this command in terminal to install
> pip install gensim

We will use fake and real news dataset to do our expirament. You can find the dataset here: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset?select=True.csv


**Runnable example:** The original news CSV was not supplied. I use a small, self-contained corpus below to demonstrate the API; I train the assessed model on the full supplied Simpsons dataset in Task 3. These example vectors are only an API demonstration.

In [9]:
# Imports are defined above. preserve_line=True avoids a Punkt download.
# Each input string is already one document/dialogue line.

In [10]:
df = pd.DataFrame({"text": [
    "the program will provide support for students",
    "schools provide a program for learning",
    "the training program will provide useful skills",
    "students learn useful skills at schools",
    "the course offers support and training",
]})
df.head()

,text
0,the program will provide support for students
1,schools provide a program for learning
2,the training program will provide useful skills
3,students learn useful skills at schools
4,the course offers support and training


In [11]:
tokens = [word_tokenize(text.lower(), preserve_line=True) for text in df["text"]]

In [12]:
w2v = gensim.models.Word2Vec(
    tokens, min_count=1, vector_size=100, window=5, sg=1,
    seed=42, workers=1, hashfxn=stable_hash,
)

In [13]:
print("Cosine similarity between 'provide' and 'program' - Skip Gram : ",w2v.wv.similarity('provide', 'program'))

Cosine similarity between 'provide' and 'program' - Skip Gram :  0.09112272


In [14]:
print("words that similar to 'program' - Skip Gram : ",w2v.wv.most_similar('program'))

words that similar to 'program' - Skip Gram :  [('a', 0.20505386590957642), ('will', 0.17067979276180267), ('for', 0.15298810601234436), ('provide', 0.09112270176410675), ('offers', 0.07083001732826233), ('schools', 0.026320042088627815), ('useful', 0.0052142757922410965), ('the', -0.002887059235945344), ('learn', -0.01590736210346222), ('learning', -0.02505647949874401)]


# Tasks

### Task 1: Cosine Similarity
Use the Cosine Similarity method to determine how similar the following sentences are.

'This is the first document.',  
'This document is the second document.',  
'And this is the third one.',  
'Is this the first document?'  


In [15]:
sentences = [
    "This is the first document.",
    "This document is the second document.",
    "And this is the third one.",
    "Is this the first document?",
]
task1_vectorizer = TfidfVectorizer()
sentence_vectors = task1_vectorizer.fit_transform(sentences)
similarities = cosine_similarity(sentence_vectors)
labels = [f"Sentence {i}" for i in range(1, 5)]
display(pd.DataFrame(similarities, index=labels, columns=labels).round(4))
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        print(f"Sentence {i+1} vs Sentence {j+1}: {similarities[i, j]:.4f}")
assert np.allclose(similarities, similarities.T)
assert np.allclose(np.diag(similarities), 1)
assert np.isclose(similarities[0, 3], 1)

,Sentence 1,Sentence 2,Sentence 3,Sentence 4
Sentence 1,1.0000,0.6469,0.3078,1.0000
Sentence 2,0.6469,1.0000,0.2252,0.6469
Sentence 3,0.3078,0.2252,1.0000,0.3078
Sentence 4,1.0000,0.6469,0.3078,1.0000


Sentence 1 vs Sentence 2: 0.6469
Sentence 1 vs Sentence 3: 0.3078
Sentence 1 vs Sentence 4: 1.0000
Sentence 2 vs Sentence 3: 0.2252
Sentence 2 vs Sentence 4: 0.6469
Sentence 3 vs Sentence 4: 0.3078


I represented each sentence with TF-IDF and compared all pairs. Sentences 1 and 4 score **1.0** because lowercasing and tokenization give them the same word counts; this representation ignores word order and punctuation. A high score indicates shared weighted vocabulary, rather than guaranteed equivalence in meaning.

### Task 2: TF-IDF
Use tf-idf method on the sentences below to determine the important words.

'data science is one of the most important fields of science',  
'this is one of the best data science courses',  
'data scientists analyze data'  


In [16]:
documents = [
    "data science is one of the most important fields of science",
    "this is one of the best data science courses",
    "data scientists analyze data",
]
# Keep default settings to show the effect of all words, including stop words.
task2_vectorizer = TfidfVectorizer()
task2_vectors = task2_vectorizer.fit_transform(documents)
tfidf_scores = pd.DataFrame(
    task2_vectors.toarray(),
    columns=task2_vectorizer.get_feature_names_out(),
    index=["Document 1", "Document 2", "Document 3"],
)
display(tfidf_scores.round(4))
for label, row in tfidf_scores.iterrows():
    ranked = row[row > 0].sort_values(ascending=False, kind="stable")
    print(f"\n{label} — words ranked by TF-IDF:")
    print(", ".join(f"{word}: {score:.4f}" for word, score in ranked.items()))
    highest = row.index[np.isclose(row.to_numpy(), row.max())].tolist()
    print("Highest-scoring word(s):", ", ".join(highest))

,analyze,best,courses,data,fields,important,is,most,of,one,science,scientists,the,this
Document 1,0.0000,0.0000,0.0000,0.1895,0.3209,0.3209,0.2440,0.3209,0.4881,0.2440,0.4881,0.0000,0.2440,0.0000
Document 2,0.0000,0.4003,0.4003,0.2364,0.0000,0.0000,0.3044,0.0000,0.3044,0.3044,0.3044,0.0000,0.3044,0.4003
Document 3,0.5427,0.0000,0.0000,0.6411,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.5427,0.0000,0.0000



Document 1 — words ranked by TF-IDF:
of: 0.4881, science: 0.4881, fields: 0.3209, important: 0.3209, most: 0.3209, is: 0.2440, one: 0.2440, the: 0.2440, data: 0.1895
Highest-scoring word(s): of, science

Document 2 — words ranked by TF-IDF:
best: 0.4003, courses: 0.4003, this: 0.4003, is: 0.3044, of: 0.3044, one: 0.3044, science: 0.3044, the: 0.3044, data: 0.2364
Highest-scoring word(s): best, courses, this

Document 3 — words ranked by TF-IDF:
data: 0.6411, analyze: 0.5427, scientists: 0.5427
Highest-scoring word(s): data


I ranked every nonzero score and included ties. With the default vectorizer, **of** and **science** tie for the highest score in Document 1, **best** and **courses** tie in Document 2, and **data** has the highest score in Document 3. Stop words remain because I used the default settings. Scikit-learn uses smoothed IDF, `log((1 + N)/(1 + df)) + 1`, raw term counts, and L2 normalization by default; this differs from the simplified formula shown above.

## Word2vec

### Task 3:

Download the Simpsons dataset **(simpsons_script_lines.csv)** and apply the preprocessing procedure.  
Use the **'spoken_words'** column.
```
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[0-9]", '', text)
    text = re.sub(r"[)(,”“.’$-]", '', text)
    return text
```
Create a skip gram Word2Vec model as below.
```
Skip_gram_model = gensim.models.Word2Vec(tokens, min_count = 1, vector_size = 100, window = 5, sg = 1)
```


In [17]:
dataset_path = Path("simpsons_script_lines.csv")
if not dataset_path.is_file():
    raise FileNotFoundError("Place simpsons_script_lines.csv beside this notebook, or upload it to the Colab working directory.")
simpsons = pd.read_csv(dataset_path)
if "spoken_words" not in simpsons.columns:
    raise ValueError("The dataset must contain a spoken_words column.")

# Apply the exact cleaning procedure requested in the lab.
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[0-9]", "", text)
    text = re.sub(r"[)(,”“.’$-]", "", text)
    return text

spoken_words = simpsons["spoken_words"].dropna().astype(str)
cleaned_lines = spoken_words.map(clean_text)
tokens = [word_tokenize(line, preserve_line=True) for line in cleaned_lines]
tokens = [line for line in tokens if line]
assert tokens, "No usable dialogue remains after preprocessing."
Skip_gram_model = gensim.models.Word2Vec(
    tokens, min_count=1, vector_size=100, window=5, sg=1,
    seed=42, workers=1, epochs=5, hashfxn=stable_hash,
)
print(f"CSV rows: {len(simpsons):,}")
print(f"Missing spoken_words: {simpsons['spoken_words'].isna().sum():,}")
print(f"Nonempty tokenized dialogue lines: {len(tokens):,}")
print(f"Total tokens: {sum(map(len, tokens)):,}")
print(f"Vocabulary size: {len(Skip_gram_model.wv):,}")
print(f"Vector dimensions: {Skip_gram_model.vector_size}")
display(pd.DataFrame({"original": spoken_words.head(3).tolist(),
                      "cleaned": cleaned_lines.head(3).tolist()}))
print("First tokenized line:", tokens[0])
required_words = {"homer", "marge", "bart", "jimbo", "milhouse", "kearney", "nelson", "patty", "selma"}
assert required_words.issubset(Skip_gram_model.wv.key_to_index)
assert Skip_gram_model.sg == 1 and Skip_gram_model.vector_size == 100

CSV rows: 158,314
Missing spoken_words: 26,459
Nonempty tokenized dialogue lines: 131,839
Total tokens: 1,481,008
Vocabulary size: 44,175
Vector dimensions: 100


,original,cleaned
0,"No, actually, it was a little of both. Sometim...",no actually it was a little of both sometimes ...
1,Where's Mr. Bergstrom?,where's mr bergstrom?
2,I don't know. Although I'd sure like to talk t...,i don't know although i'd sure like to talk to...


First tokenized line: ['no', 'actually', 'it', 'was', 'a', 'little', 'of', 'both', 'sometimes', 'when', 'a', 'disease', 'is', 'in', 'all', 'the', 'magazines', 'and', 'all', 'the', 'news', 'shows', 'it', "'s", 'only', 'natural', 'that', 'you', 'think', 'you', 'have', 'it']


I used the full supplied CSV and only its `spoken_words` column for training. I removed missing entries before cleaning, applied the supplied regex rules exactly, and discarded empty token lists. I kept punctuation that the supplied rules leave behind. I used NLTK tokenization with `preserve_line=True` because each row is already one dialogue line. The required Skip-gram parameters are unchanged; I added reproducibility settings and made the default five training epochs explicit.

### Task 4

Use: wv.most_similar() method to :

1.	Find the words similar to “homer”.

2.	Find the words similar to “marge”.

3. Find the words similar to “bart”



In [18]:
similar_word_results = {}
for word in ["homer", "marge", "bart"]:
    neighbors = Skip_gram_model.wv.most_similar(word, topn=10)
    similar_word_results[word] = neighbors
    print(f"Words most similar to '{word}':")
    display(pd.DataFrame(neighbors, columns=["word", "cosine_similarity"]).round(4))

Words most similar to 'homer':


,word,cosine_similarity
0,abe,0.8956
1,marge,0.8535
2,bart,0.8531
3,eliza,0.8450
4,bartholomew,0.8416
5,grampa,0.8276
6,allison,0.8215
7,kirk,0.8202
8,waylon,0.8154
9,sweetheart,0.8151


Words most similar to 'marge':


,word,cosine_similarity
0,abe,0.8756
1,homer,0.8535
2,becky,0.8433
3,sweetheart,0.8374
4,sweetie,0.8368
5,midge,0.8311
6,allison,0.8236
7,eliza,0.8223
8,lisa,0.8217
9,luann,0.8197


Words most similar to 'bart':


,word,cosine_similarity
0,lisa,0.8762
1,milhouse,0.8602
2,eliza,0.8559
3,homer,0.8531
4,abe,0.8473
5,grampa,0.8473
6,allison,0.8438
7,jessica,0.8431
8,sweetie,0.8341
9,maggie,0.8232


I used `wv.most_similar()` to return the ten nearest words for each requested name. These scores compare learned word vectors. Neighbors reflect usage in this dialogue corpus and are not necessarily synonyms or family relationships.

### Task 5

Use the wv.doesnt_match() method to :

1.	Find which of 'jimbo', 'milhouse’, and 'kearney’ does not belong to the list.

3.	Find the odd one among "nelson", "bart", and "milhouse".

4.	Find the odd one among ‘homer', 'patty', and ‘selma'.

*Hint: You need to pass the strings as List*

In [19]:
word_groups = [
    ["jimbo", "milhouse", "kearney"],
    ["nelson", "bart", "milhouse"],
    ["homer", "patty", "selma"],
]
odd_word_results = []
for group in word_groups:
    odd_word = Skip_gram_model.wv.doesnt_match(group)
    odd_word_results.append({"word_group": ", ".join(group), "odd_word": odd_word})
display(pd.DataFrame(odd_word_results))

,word_group,odd_word
0,"jimbo, milhouse, kearney",milhouse
1,"nelson, bart, milhouse",nelson
2,"homer, patty, selma",homer


I passed each requested group as a list to `wv.doesnt_match()`. It returns the word least similar to the group's average normalized vector. I report the trained model's output rather than assuming an answer from character relationships. Results can vary with preprocessing, corpus, training settings, and library versions.